# Credit Risk Analysis — Predicting Loan Default

**Author:** Bekir Onal · [GitHub](https://github.com/bekir35-on) · [LinkedIn](https://www.linkedin.com/in/bekir-onal-636b57250)

This notebook explores a credit portfolio of **37,408 loans** to answer a practical business question: *which loans are likely to default, and what drives that risk?*

**Approach:** data cleaning → exploratory data analysis (EDA) → segment analysis → visualisation → machine-learning models (Logistic Regression & Decision Tree) → feature importance → business recommendations.

**Tools:** Python · pandas · scikit-learn · Matplotlib


## 1. Load the data


In [ ]:
import pandas as pd
import glob

# train dosyasini otomatik bul (isimdeki bosluk yuzunden elle yol yazmiyoruz)
train_path = glob.glob('/kaggle/input/**/*train-FIN*', recursive=True)[0]
df = pd.read_excel(train_path)   # eski .xls -> read_excel

print("Rows, Columns:", df.shape)
print(list(df.columns))
df.head()


## 2. Understand the columns and missing data


In [ ]:
df.info()
print(df['QUALITY_OF_LOAN'].value_counts())
for col in ['INF_GENDER','INF_MARITAL_STATUS','COMPENSATION_CHARGED','CLIENT_TYPE','REPAY_MODE']:
    print(df[col].value_counts())


## 3. Target and overall default rate


In [ ]:
# Bad (kotu) = 1, Good (iyi) = 0
df['DEFAULT'] = (df['QUALITY_OF_LOAN'] == 'B').astype(int)
print("Overall default rate: {:.1%}".format(df['DEFAULT'].mean()))


## 4. Segment analysis — where is the risk highest?


In [ ]:
for col in ['COMPENSATION_CHARGED','REPAY_MODE','CLIENT_TYPE','INF_MARITAL_STATUS','INF_GENDER']:
    seg = df.groupby(col)['DEFAULT'].agg(['mean','count']).sort_values('mean', ascending=False)
    seg['mean'] = (seg['mean']*100).round(1)
    print(col); print(seg)


## 5. Visualisation


In [ ]:
import matplotlib.pyplot as plt

client = df[df['CLIENT_TYPE'].isin(['Rural','Semi-urban','Urban'])]
by_client = client.groupby('CLIENT_TYPE')['DEFAULT'].mean().sort_values()*100
by_comp   = df.groupby('COMPENSATION_CHARGED')['DEFAULT'].mean().sort_values()*100
overall   = df['DEFAULT'].mean()*100

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].barh(by_client.index, by_client.values, color='#4C78A8')
axes[0].axvline(overall, color='#E4572E', linestyle='--')
axes[0].set_title('Default Rate by Client Type'); axes[0].set_xlabel('Default rate (%)')
axes[1].barh(by_comp.index, by_comp.values, color='#4C78A8')
axes[1].axvline(overall, color='#E4572E', linestyle='--')
axes[1].set_title('Default Rate by Compensation Charged'); axes[1].set_xlabel('Default rate (%)')
plt.tight_layout(); plt.show()


## 6. Prepare data for modelling


In [ ]:
y = df['DEFAULT']
X = df.drop(columns=['ACC_NO','DEFAULT','QUALITY_OF_LOAN'])   # kimlik + leakage cikar

num_cols = X.select_dtypes(include='number').columns
cat_cols = X.select_dtypes(include='object').columns
for c in num_cols: X[c] = X[c].fillna(X[c].median())
for c in cat_cols: X[c] = X[c].fillna(X[c].mode()[0])
X = pd.get_dummies(X, columns=list(cat_cols), drop_first=True)   # one-hot
print("Feature matrix shape:", X.shape)


## 7. Models: Logistic Regression & Decision Tree


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
Xtr = scaler.fit_transform(X_train); Xte = scaler.transform(X_test)
logreg = LogisticRegression(max_iter=1000).fit(Xtr, y_train)
auc_lr = roc_auc_score(y_test, logreg.predict_proba(Xte)[:,1])

tree = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train, y_train)
auc_dt = roc_auc_score(y_test, tree.predict_proba(X_test)[:,1])

print("Logistic Regression ROC-AUC:", round(auc_lr, 3))
print("Decision Tree ROC-AUC:      ", round(auc_dt, 3))


## 8. Feature importance


In [ ]:
imp = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
print((imp*100).round(1))


## Conclusion & Business Recommendations

**Key findings**
- Overall default rate: **11.1%** (4,154 of 37,408 loans).
- Highest-risk segments: a **compensation/penalty charged** (16.1% vs 7.0%) and **rural** customers (12.8% vs 1.3% for urban).

**Models** — Logistic Regression ROC-AUC **0.671**, Decision Tree ROC-AUC **0.676**: a real but limited signal given only 11 raw features. ROC-AUC is used instead of accuracy because the data is imbalanced (~11% default).

**Feature importance** — `COMPENSATION_CHARGED` (41.7%), `ACCCURRENTBALANCE` (19.2%) and `CLIENT_TYPE = Urban` (16.3%) account for ~77% of the model's decisions.

**Business recommendations**
1. Flag applicants with a **prior penalty charge** for stricter review — the strongest single signal.
2. Use **region** and **current balance** as supporting factors in credit scoring.
3. Collect **richer data** (credit history, income, past repayment behaviour) to push predictive power beyond ~0.68.

**Fairness note:** gender showed almost no predictive power and should **not** be used as a risk factor.
